# 🧪 RAG-Anything — RAGAS Evaluation Notebook

Notebook này đánh giá hệ thống RAG-Anything của bạn sử dụng:
- **Dataset**: `thangvip/vietnamese-legal-qa` (HuggingFace)
- **Framework đánh giá**: RAGAS
- **Metrics**: Faithfulness, Answer Relevance, Context Recall, Context Precision

> ⚠️ **Yêu cầu**: API server RAG-Anything phải đã chạy và public URL (ngrok) phải sẵn sàng trước khi chạy phần Evaluation.

---
## 🔧 PHẦN 1: Khởi động API Server RAG-Anything

In [ ]:
# ============================================================
# 1.1 — Cài đặt dependencies
# ============================================================
!git clone https://github.com/cuonglevan23/RAG-anything-fastapi
%cd RAG-anything-fastapi

!pip install -q raganything nest_asyncio pyngrok loguru
!pip install -q 'mineru[all]>=2.7.0'
!pip install -q transformers accelerate

In [ ]:
# ============================================================
# 1.2 — Cấu hình API Keys
# ============================================================
import os

# 🔑 THAY THẾ các giá trị bên dưới:
os.environ['OPENAI_API_KEY']   = 'sk-...YOUR_OPENAI_KEY...'    # Bắt buộc
os.environ['LLM_MODEL']        = 'gpt-4o-mini'
os.environ['EMBEDDING_MODEL']  = 'text-embedding-3-small'

# Cohere Reranker (tùy chọn - bỏ qua nếu chưa có)
# os.environ['RERANK_ENABLE']   = 'true'
# os.environ['COHERE_API_KEY']  = 'co-...YOUR_COHERE_KEY...'

print('✅ API Keys đã được cấu hình.')

In [ ]:
# ============================================================
# 1.3 — Khởi động ngrok + FastAPI server
# ============================================================
from pyngrok import ngrok
import uvicorn
import nest_asyncio
import threading

NGROK_AUTH_TOKEN = '2o9DM9nRRPK15EyjBzCIdQlZsfX_7K36VXxLCXspU8aYrjBPi'  # Token của bạn
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(8000)
API_BASE_URL = str(public_url).replace('NgrokTunnel: ', '').split(' ')[0].strip('"')
print(f'🌐 API đang chạy tại: {API_BASE_URL}')
print(f'📌 Swagger UI: {API_BASE_URL}/docs')

nest_asyncio.apply()

from app.main import app

def run_server():
    config = uvicorn.Config(app, host='0.0.0.0', port=8000, loop='asyncio')
    server = uvicorn.Server(config)
    import asyncio
    asyncio.run(server.serve())

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

import time
time.sleep(5)
print('✅ Server started!')

---
## 📂 PHẦN 2: Upload tài liệu vào RAG System

> Upload file PDF Luật Thư Viện (hoặc bất kỳ văn bản pháp luật nào) vào project để index.

In [ ]:
# ============================================================
# 2.1 — Upload file PDF và đợi xử lý xong
# ============================================================
import requests
import time

API_PREFIX  = '/api/v1/rag'
PROJECT_ID  = 'legal_eval'           # Tên project evaluation
PDF_PATH    = '/content/luat_thu_vien.pdf'  # ← ĐỔI ĐƯỜNG DẪN FILE PDF CỦA BẠN

api_url = f'{API_BASE_URL}{API_PREFIX}'

def upload_and_wait(pdf_path, project_id):
    print(f'📤 Uploading: {pdf_path}...')
    with open(pdf_path, 'rb') as f:
        resp = requests.post(
            f'{api_url}/upload',
            files={'file': (os.path.basename(pdf_path), f, 'application/pdf')},
            data={'project_id': project_id}
        )
    if resp.status_code != 200:
        raise Exception(f'Upload failed: {resp.text}')

    task_id = resp.json()['task_id']
    print(f'📋 Task ID: {task_id} — Đang chờ xử lý...')

    while True:
        s = requests.get(f'{api_url}/status/{task_id}').json()
        pct    = s.get('percentage', 0)
        status = s.get('status')
        msg    = s.get('message', '')
        print(f'  [{pct:.0f}%] {status}: {msg}', end='\r')

        if status == 'completed':
            print(f'\n✅ Xử lý hoàn tất!')
            break
        elif status == 'failed':
            raise Exception(f'Processing failed: {s.get("error")}')
        time.sleep(3)

upload_and_wait(PDF_PATH, PROJECT_ID)

---
## 📊 PHẦN 3: Load Dataset & Chạy RAG Queries

> Tải dataset `thangvip/vietnamese-legal-qa` từ HuggingFace, query hệ thống RAG, thu thập contexts.

In [ ]:
# ============================================================
# 3.1 — Cài đặt RAGAS + load dataset
# ============================================================
!pip install -q ragas datasets langchain-openai

from datasets import load_dataset

# Load dataset Vietnamese Legal QA
print('📦 Loading Vietnamese Legal QA dataset...')
raw_dataset = load_dataset('thangvip/vietnamese-legal-qa', split='train')
print(f'✅ Loaded {len(raw_dataset)} samples')
print(f'Columns: {raw_dataset.column_names}')
raw_dataset[0]  # Xem mẫu đầu tiên

In [ ]:
# ============================================================
# 3.2 — Chọn N mẫu để evaluate (bắt đầu nhỏ để tiết kiệm cost)
# ============================================================
N_SAMPLES    = 30       # ← Tăng lên nếu muốn evaluate nhiều hơn
QUERY_MODE   = 'hybrid'  # hybrid | local | global | naive | mix

# Lấy N mẫu ngẫu nhiên
import random
random.seed(42)
indices = random.sample(range(len(raw_dataset)), N_SAMPLES)
eval_samples = raw_dataset.select(indices)

print(f'📝 Sẽ evaluate {N_SAMPLES} câu hỏi với mode={QUERY_MODE}')

In [ ]:
# ============================================================
# 3.3 — Query RAG system và thu thập retrieved_contexts
# ============================================================
import json
from tqdm import tqdm

def query_rag(question, mode=QUERY_MODE, project_id=PROJECT_ID):
    """Gọi API query và trả về (answer, contexts)"""
    payload = {
        'query':      question,
        'mode':       mode,
        'project_id': project_id
    }
    try:
        resp = requests.post(f'{api_url}/query', json=payload, timeout=120)
        if resp.status_code == 200:
            answer = resp.json().get('response', '')
            return answer
        else:
            return f'[API Error {resp.status_code}]'
    except Exception as e:
        return f'[Request Error: {e}]'

# Xác định tên cột question/answer từ dataset
# Kiểm tra các tên cột có thể có
sample = eval_samples[0]
q_col = 'question' if 'question' in sample else list(sample.keys())[0]
a_col = 'answer'   if 'answer'   in sample else list(sample.keys())[1]
print(f'Question column: "{q_col}", Answer column: "{a_col}"')

# Chạy queries
results = []
for i, sample in enumerate(tqdm(eval_samples, desc='🔍 Querying RAG')):
    question     = sample[q_col]
    ground_truth = sample[a_col]
    answer       = query_rag(question)

    results.append({
        'question':     question,
        'answer':       answer,         # Câu trả lời từ hệ thống RAG
        'ground_truth': ground_truth,   # Câu trả lời chuẩn từ dataset
        'contexts':     [answer],       # RAGAS cần contexts; dùng answer làm proxy nếu API ko trả contexts riêng
    })

print(f'\n✅ Collected {len(results)} results')
print(f'\nSample result:\n  Q: {results[0]["question"][:100]}\n  A: {results[0]["answer"][:100]}')

---
## 🎯 PHẦN 4: Đánh giá với RAGAS

In [ ]:
# ============================================================
# 4.1 — Chuẩn bị RAGAS Dataset
# ============================================================
from datasets import Dataset as HFDataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
)
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Tạo RAGAS dataset
ragas_dataset = HFDataset.from_list([
    {
        'question':     r['question'],
        'answer':       r['answer'],
        'contexts':     r['contexts'],   # List[str]
        'ground_truth': r['ground_truth'],
    }
    for r in results
])

print(f'📊 RAGAS dataset ready: {len(ragas_dataset)} samples')
ragas_dataset

In [ ]:
# ============================================================
# 4.2 — Chạy RAGAS Evaluation
# ============================================================
# RAGAS dùng LLM để chấm điểm → cần OpenAI API key
llm        = ChatOpenAI(model='gpt-4o-mini', openai_api_key=os.environ['OPENAI_API_KEY'])
embeddings = OpenAIEmbeddings(model='text-embedding-3-small', openai_api_key=os.environ['OPENAI_API_KEY'])

print('🚀 Đang chạy RAGAS evaluation...')
print('(Mỗi sample cần 3-5 LLM calls → ước tính ~2-5 phút cho 30 samples)\n')

result = evaluate(
    dataset = ragas_dataset,
    metrics = [
        faithfulness,       # Câu trả lời có trung thực với context không?
        answer_relevancy,   # Câu trả lời có liên quan tới câu hỏi không?
        context_recall,     # Retrieved context có đủ thông tin để trả lời không?
        context_precision,  # Context có nhiều thông tin nhiễu không?
    ],
    llm         = llm,
    embeddings  = embeddings,
    raise_exceptions = False,
)

print('\n✅ Evaluation hoàn tất!')

In [ ]:
# ============================================================
# 4.3 — Xem kết quả tổng hợp
# ============================================================
import pandas as pd

print('=' * 55)
print('           📊 RAGAS EVALUATION RESULTS')
print('=' * 55)

metrics_data = {
    'Metric':      ['Faithfulness', 'Answer Relevancy', 'Context Recall', 'Context Precision'],
    'Score':       [
        result['faithfulness'],
        result['answer_relevancy'],
        result['context_recall'],
        result['context_precision'],
    ],
    'Ý nghĩa': [
        'Model có bịa thông tin không có trong tài liệu không?',
        'Câu trả lời có đúng trọng tâm câu hỏi không?',
        'Hệ thống có retrieved đủ nội dung cần thiết không?',
        'Context retrieved có nhiều thông tin thừa không?',
    ]
}

df = pd.DataFrame(metrics_data)
df['Score'] = df['Score'].apply(lambda x: f'{x:.4f} ({x*100:.1f}%)')
print(df.to_string(index=False))
print('=' * 55)

# Overall score
scores = [result['faithfulness'], result['answer_relevancy'],
          result['context_recall'], result['context_precision']]
overall = sum(scores) / len(scores)
print(f'\n🏆 Overall Score: {overall:.4f} ({overall*100:.1f}%)')

# Phân loại chất lượng
if overall >= 0.8:
    grade = '🟢 Excellent'
elif overall >= 0.6:
    grade = '🟡 Good'
elif overall >= 0.4:
    grade = '🟠 Fair'
else:
    grade = '🔴 Poor — cần cải thiện'
print(f'📈 Grade: {grade}')

In [ ]:
# ============================================================
# 4.4 — Xem chi tiết từng sample (debug các câu trả lời tệ)
# ============================================================
detail_df = result.to_pandas()

# Sắp xếp theo faithfulness thấp nhất để debug
detail_df_sorted = detail_df.sort_values('faithfulness', ascending=True)
print('📉 Top 5 câu hỏi có Faithfulness thấp nhất (cần lưu ý):')
detail_df_sorted[['question', 'faithfulness', 'answer_relevancy']].head(5)

In [ ]:
# ============================================================
# 4.5 — So sánh các Query Mode (tùy chọn — tốn thêm cost)
# ============================================================
# Bỏ comment block này nếu muốn so sánh mode hybrid vs local vs naive

# MODES_TO_COMPARE = ['hybrid', 'local', 'naive']
# comparison = {}

# for mode in MODES_TO_COMPARE:
#     print(f'\n🔄 Evaluating mode: {mode}...')
#     mode_results = []
#     for sample in tqdm(eval_samples, desc=f'  Querying [{mode}]'):
#         question     = sample[q_col]
#         ground_truth = sample[a_col]
#         answer       = query_rag(question, mode=mode)
#         mode_results.append({
#             'question': question, 'answer': answer,
#             'ground_truth': ground_truth, 'contexts': [answer]
#         })

#     mode_dataset = HFDataset.from_list(mode_results)
#     mode_eval    = evaluate(dataset=mode_dataset, metrics=[faithfulness, answer_relevancy],
#                             llm=llm, embeddings=embeddings, raise_exceptions=False)
#     comparison[mode] = {
#         'faithfulness':    mode_eval['faithfulness'],
#         'answer_relevancy': mode_eval['answer_relevancy'],
#     }

# print('\n📊 Mode Comparison:')
# pd.DataFrame(comparison).T

In [ ]:
# ============================================================
# 4.6 — Lưu kết quả ra CSV
# ============================================================
output_path = f'/content/ragas_results_mode_{QUERY_MODE}.csv'
detail_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'💾 Kết quả đã được lưu tại: {output_path}')

# Download về máy
from google.colab import files
files.download(output_path)

---
## 📖 Hướng dẫn đọc kết quả

| Metric | Điểm lý tưởng | Điểm thấp nghĩa là... |
|---|---|---|
| **Faithfulness** | > 0.8 | Model đang **bịa** thông tin không có trong tài liệu |
| **Answer Relevancy** | > 0.8 | Câu trả lời đang **lạc đề** so với câu hỏi |
| **Context Recall** | > 0.7 | Hệ thống **không retrieve đủ** ngữ cảnh cần thiết → giảm chunk_size |
| **Context Precision** | > 0.7 | Retrieved quá nhiều chunk **không liên quan** → bật reranker |

### 🔧 Hành động cải thiện:
- **Context Recall thấp** → Giảm `chunk_token_size` (đã làm: 600), dùng mode `local`
- **Context Precision thấp** → Bật **Cohere Reranker** (đã cấu hình sẵn)
- **Faithfulness thấp** → Cải thiện extraction prompt, tăng `entity_extract_max_gleaning`
- **Answer Relevancy thấp** → Thử `response_type` khác trong `aquery()`